The purpose of this notebook is to verify the correctness of the functions in `utils_qubit_hamiltonian.py`, and present simple examples of their usage

In [ ]:
import sys
sys.path.append('../')

import numpy as np
from utils_hamiltonian_qubit import (
    split_pauli_operator,
    pauli_matrix_element_with_basis_state,
    remove_irrelevant_pauli_terms,
    taper_hamiltonian,
    CNOT_clifford_operator,
    seniority_solving_clifford_operator,
    append_seniority_solving_clifford_circuit,
    group_odds_and_evens,
    ungroup_odds_and_evens,
    process_hamiltonian_to_remove_irrelevant_terms,
    process_hamiltonian_to_project_onto_symmetric_subspace
)
from openfermion import (
    QubitOperator,
    get_sparse_operator,
    get_fermion_operator,
    MolecularData,
    jordan_wigner
)
from openfermionpyscf import run_pyscf
from utils_basic import (
    random_pauli_term, 
    random_bin_list, 
    state_from_bin_list,
    compute_product_of_unitaries,
    random_pauli_hamiltonian,
    apply_unitary_product,
    compute_product_of_unitaries
)
from utils_misc import (
    duplicate_pauli_term_qubit_wise,
    duplicate_hamiltonian_qubit_wise,
    duplicate_statevector_qubit_wise
)
from utils_fc import (
    decimal_to_binary_list
)
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator
from numpy.random import uniform
import random

## Pauli Operator Matrix Elements

Here, I verify the following:
- `split_pauli_operator` should cleave a Pauli operator into two Pauli operators correctly


In [2]:
P = QubitOperator('X0 X1 X2 X3 X4 Y5 Y6 Y7 Y8 Y9')
N = 5
P0, P1 = split_pauli_operator(P, 5)

print(f'''
    Original Pauli : {P}
    Left Split     : {QubitOperator(P0)}
    Right Split    : {QubitOperator(P1)}
''')


P = QubitOperator('X0 Y5 Y6 Y7 Y8 Y9')
N = 3
P0, P1 = split_pauli_operator(P, 5)

print(f'''
    Original Pauli : {P}
    Left Split     : {QubitOperator(P0)}
    Right Split    : {QubitOperator(P1)}
''')


    Original Pauli : 1.0 [X0 X1 X2 X3 X4 Y5 Y6 Y7 Y8 Y9]
    Left Split     : 1.0 [X0 X1 X2 X3 X4]
    Right Split    : 1.0 [Y5 Y6 Y7 Y8 Y9]


    Original Pauli : 1.0 [X0 Y5 Y6 Y7 Y8 Y9]
    Left Split     : 1.0 [X0]
    Right Split    : 1.0 [Y5 Y6 Y7 Y8 Y9]



Here, I verify the following:

- <v|P|w> is calculated correctly for a bunch of P, v, w examples by comparing with numerical value

In [3]:
for Nqubits in range(2, 12):
    for _ in range(40):
        _, P = random_pauli_term(Nqubits)
        v    = random_bin_list(Nqubits)
        w    = random_bin_list(Nqubits)

        matrix_element_numerical  = state_from_bin_list(v) @ get_sparse_operator(P, Nqubits) @ state_from_bin_list(w)
        matrix_element_analytical = pauli_matrix_element_with_basis_state(P, v, w)
        
        print(np.abs(matrix_element_numerical - matrix_element_analytical) < 1e-12, end='')
    print()

TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrueTrue
TrueTrueTrueTrueTrueTrueTrueTrueTr

## Hamiltonian modification based on symmetries

First, I generate a simple non-trivial Hamiltonian to work with, and implement the term deletion and tapering based on computational basis states.

In [55]:
geo = [
    ['H', [0,0,0]],
    ['H', [0,0,1]],
    ['H', [0,0,2]],
    ['H', [0,0,3]]
]
mol = run_pyscf(
    MolecularData(geo, 'sto3g', 1, 0), 
    run_scf=True
)
H = jordan_wigner(get_fermion_operator(mol.get_molecular_hamiltonian()))

# play around with this example by modifying v and w
v = [1,0,0,1]
w = [0,1,1,0]

Hrel     = remove_irrelevant_pauli_terms(H, v, w)
Htapered = taper_hamiltonian(H, v, w)

print(f'''
    Number of terms in H        : {len(H.terms)}
    Number of terms in Hrel     : {len(Hrel.terms)}
    Number of terms in Htapered : {len(Htapered.terms)}
''')

print('Hrel')
print(Hrel)
print()

print('Htapered')
print(Htapered)

print()
print(Htapered - taper_hamiltonian(Hrel, v, w))



    Number of terms in H        : 185
    Number of terms in Hrel     : 4
    Number of terms in Htapered : 1

Hrel
(-0.039345513560407545+0j) [X0 X1 Y2 Y3] +
(0.039345513560407545+0j) [X0 Y1 Y2 X3] +
(0.039345513560407545+0j) [Y0 X1 X2 Y3] +
(-0.039345513560407545+0j) [Y0 Y1 X2 X3]

Htapered
(-0.15738205424163018+0j) []

0


The purpose of this code is to modify the Hamiltonian without changing the relevant matrix element and/or expectation values. So we should see if that works as desired.

Here, I see if, for states $|\Psi_s\rangle = |s\rangle |\psi_s\rangle$, and $|\Psi_t\rangle = |t\rangle |\psi_t\rangle$, we have

$$
\langle \Psi_s|H|\Psi_t\rangle = \langle \Psi_s|H_{\text{rel}}|\Psi_t\rangle = \langle \psi_s|H_\text{tapered}|\psi_t \rangle
$$

I will use the LiH Hamiltonian since it is larger

In [ ]:
geo = [
    ['H', [0,0,0]],
    ['Li', [0,0,1]]
]
mol     = run_pyscf(MolecularData(geo, 'sto3g', 1, 0), run_scf=True)
H       = jordan_wigner(get_fermion_operator(mol.get_molecular_hamiltonian()))
Nqubits = 12

for _ in range(20):
    taper_size = random.sample(range(2, 8), 1)[0]

    bin_s = random_bin_list(taper_size)
    bin_t = random_bin_list(taper_size)

    ket_s = state_from_bin_list(bin_s, taper_size)
    ket_t = state_from_bin_list(bin_t, taper_size)

    psi_s = uniform(-1, 1, 2 ** (Nqubits - taper_size))
    psi_t = uniform(-1, 1, 2 ** (Nqubits - taper_size))

    Psi_s = np.kron(ket_s, psi_s)
    Psi_t = np.kron(ket_t, psi_t)

    Hsparse         = get_sparse_operator(H, Nqubits)
    ev_full         = Psi_s @ Hsparse @ Psi_t

    Hrel_sparse     = get_sparse_operator(remove_irrelevant_pauli_terms(H, bin_s, bin_t), Nqubits)
    ev_rel          = Psi_s @ Hrel_sparse @ Psi_t

    Htapered_sparse = get_sparse_operator(taper_hamiltonian(H, bin_s, bin_t), Nqubits - taper_size)
    ev_tapered      = psi_s @ Htapered_sparse @ psi_t

    print(f'''
        full expectation    : {ev_full}
        rel expectation     : {ev_rel}
        tapered expectation : {ev_tapered}

        all equal           : {np.abs(ev_full-ev_rel)<1e-12 and np.abs(ev_full-ev_tapered)<1e-12 and np.abs(ev_rel-ev_tapered)<1e-12}
    ''')


        full expectation    : (0.001332959337814405+0j)
        rel expectation     : (0.0013329593378143237+0j)
        tapered expectation : (0.0013329593378143298+0j)

        all equal           : True
    

        full expectation    : (-0.0493525065384902+0j)
        rel expectation     : (-0.0493525065384902+0j)
        tapered expectation : (-0.0493525065384902+0j)

        all equal           : True
    

        full expectation    : (-1.0243185515630386e-19+0j)
        rel expectation     : 0j
        tapered expectation : 0.0

        all equal           : True
    

        full expectation    : (-0.012704784852972481+0j)
        rel expectation     : (-0.012704784852972481+0j)
        tapered expectation : (-0.012704784852972481+0j)

        all equal           : True
    

        full expectation    : 0j
        rel expectation     : 0j
        tapered expectation : 0.0

        all equal           : True
    

        full expectation    : 0j
        rel expectation 

## Solving Seniority Operators

First, I check that $ e^{-i\pi/4} U_\text{CNOT}(0,1) = \text{CNOT}(0,1)$ and $e^{-i \pi / 4} U_\text{CNOT}(1,0) = \text{CNOT}(1,0)$

In [6]:
CNOT_01 = np.array([
    [1,0,0,0],
    [0,1,0,0],
    [0,0,0,1],
    [0,0,1,0]
])

CNOT_10 = np.array([
    [1,0,0,0],
    [0,0,0,1],
    [0,0,1,0],
    [0,1,0,0]
])

CNOT_01_from_func = get_sparse_operator(compute_product_of_unitaries(CNOT_clifford_operator(0,1)), 2).toarray()

CNOT_10_from_func = get_sparse_operator(compute_product_of_unitaries(CNOT_clifford_operator(1,0)), 2).toarray()


print(
    np.allclose(np.exp(-1j * np.pi / 4) * CNOT_01_from_func, CNOT_01), end=''
)

print(
    np.allclose(np.exp(-1j * np.pi / 4) * CNOT_10_from_func, CNOT_10)
)

TrueTrue


Now I check if the seniority solving operator and the seniority solving circuits are the same

In [7]:
for Nqubits in [2,4,6,8,10,12]:
    Uof = get_sparse_operator(compute_product_of_unitaries(seniority_solving_clifford_operator(Nqubits)), Nqubits).toarray()

    qc = QuantumCircuit(Nqubits)
    append_seniority_solving_clifford_circuit(qc, Nqubits)

    Uqk = Operator(qc).reverse_qargs().data

    print(
        np.allclose(np.exp(-1j * np.pi * (Nqubits // 2) / 4) * Uof, Uqk), end=''
    )

TrueTrueTrueTrueTrueTrue

Now, I check if $U z_{2i} z_{2i+1} U^T = z_{2i}$

In [8]:
Nqubits = 12
Ucliff  = seniority_solving_clifford_operator(Nqubits)

for i in range(Nqubits // 2):
    parity  = QubitOperator(f'Z{2*i} Z{2*i+1}')
    rotated = apply_unitary_product(parity, Ucliff)
    print(rotated == QubitOperator(f'Z{2*i}'), end='')

TrueTrueTrueTrueTrueTrue

## Full Tapering Procedure for Seniority

First: I check manually the reordering maps. First: applying both should produce the Hamiltonian I started with.

In [9]:
for Nqubits in range(2, 12):
    for Nterms in range(Nqubits // 2, 5 * Nqubits):
        H       = random_pauli_hamiltonian(Nqubits, Nterms)
        Hrecov1 = ungroup_odds_and_evens(group_odds_and_evens(H, Nqubits), Nqubits)
        Hrecov2 = group_odds_and_evens(ungroup_odds_and_evens(H, Nqubits), Nqubits)

        print(H - Hrecov1, end='')
        print(H - Hrecov2, end='')
        print(Hrecov1 - Hrecov2, end='')
    print()

000000000000000000000000000
000000000000000000000000000000000000000000
000000000000000000000000000000000000000000000000000000
000000000000000000000000000000000000000000000000000000000000000000000
000000000000000000000000000000000000000000000000000000000000000000000000000000000
000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000


Next, I check the reordering maps on some simple examples.

In [10]:
H          = QubitOperator('X0 Y1 X2 Y3 X4 Y5')
Hgrouped   = group_odds_and_evens(H, Nqubits)
Hungrouped = ungroup_odds_and_evens(Hgrouped, Nqubits)
Nqubits    = 6

print(f'''
    H          : {list(H.terms.keys())}
    Hgrouped   : {list(Hgrouped.terms.keys())}
    Hungrouped : {list(Hungrouped.terms.keys())}

    {H - Hungrouped == QubitOperator().zero()}
''')


    H          : [((0, 'X'), (1, 'Y'), (2, 'X'), (3, 'Y'), (4, 'X'), (5, 'Y'))]
    Hgrouped   : [((0, 'X'), (1, 'X'), (2, 'X'), (6, 'Y'), (7, 'Y'), (8, 'Y'))]
    Hungrouped : [((0, 'X'), (1, 'Y'), (2, 'X'), (3, 'Y'), (4, 'X'), (5, 'Y'))]

    True



Next, I check if, for a seniority conserving Hamiltonian, applying the Clifford transformation and regrouping produces zs on the first N/2 qubits

In [11]:
Nqubits = 12

H = duplicate_hamiltonian_qubit_wise(
    np.round(uniform(-2, 2), 4) * QubitOperator('X0 Y1 X2 Y3 X4 X5') + 
    np.round(uniform(-2, 2), 4) * QubitOperator('Y0 X1 Z2 X3 Z4 Y5') + 
    np.round(uniform(-2, 2), 4) * QubitOperator('Y0 Y1 X2 X3 X4 X5') + 
    np.round(uniform(-2, 2), 4) * QubitOperator('X0 X1 Z2 Y3 Z4 Z5') + 
    np.round(uniform(-2, 2), 4) * QubitOperator('Z0 X1 X2 Y3 X4 Y5') + 
    np.round(uniform(-2, 2), 4) * QubitOperator('Z0 Y1 Z2 Z3 Z4 Y5') + 
    np.round(uniform(-2, 2), 4) * QubitOperator('X0 Z1 X2 X3 Z4 Y5')
)

Ucliff = seniority_solving_clifford_operator(Nqubits)

Hrotated_temp           = apply_unitary_product(H, Ucliff)

Hrotated = QubitOperator()
for term, coef in Hrotated_temp.terms.items():
    Hrotated += np.round(coef, 4) * QubitOperator(term)

Hrotated_reordered = group_odds_and_evens(Hrotated, Nqubits)

print(H)

print()

print(Hrotated)

print()

print(Hrotated_reordered)

0.705 [X0 X1 X2 X3 Z4 Z5 Y6 Y7 Z8 Z9 Z10 Z11] +
-0.0097 [X0 X1 Y2 Y3 X4 X5 Y6 Y7 X8 X9 X10 X11] +
1.6725 [X0 X1 Z2 Z3 X4 X5 X6 X7 Z8 Z9 Y10 Y11] +
1.3214 [Y0 Y1 X2 X3 Z4 Z5 X6 X7 Z8 Z9 Y10 Y11] +
0.8608 [Y0 Y1 Y2 Y3 X4 X5 X6 X7 X8 X9 X10 X11] +
-0.1989 [Z0 Z1 X2 X3 X4 X5 Y6 Y7 X8 X9 Y10 Y11] +
1.3845 [Z0 Z1 Y2 Y3 Z4 Z5 Z6 Z7 Z8 Z9 Y10 Y11]

0.8608 [Z0 X1 Z2 X3 X5 X7 X9 X11] +
1.3214 [Z0 X1 X3 Z4 X7 Z8 Z10 X11] +
1.3845 [Z0 Z2 X3 Z4 Z6 Z8 Z10 X11] +
-0.1989 [Z0 X3 X5 Z6 X7 X9 Z10 X11] +
-0.0097 [X1 Z2 X3 X5 Z6 X7 X9 X11] +
-1.6725 [X1 Z2 X5 X7 Z8 Z10 X11] +
-0.705 [X1 X3 Z4 Z6 X7 Z8 Z10]

1.3845 [Z0 Z1 Z2 Z3 Z4 Z5 X7 X11] +
0.8608 [Z0 Z1 X6 X7 X8 X9 X10 X11] +
1.3214 [Z0 Z2 Z4 Z5 X6 X7 X9 X11] +
-0.1989 [Z0 Z3 Z5 X7 X8 X9 X10 X11] +
-0.0097 [Z1 Z3 X6 X7 X8 X9 X10 X11] +
-1.6725 [Z1 Z4 Z5 X6 X8 X9 X11] +
-0.705 [Z2 Z3 Z4 Z5 X6 X7 X9]


It is instructive to see how it maps all seniority conserving operators.

In [45]:
XX = QubitOperator('X0 X1')
YY = QubitOperator('Y0 Y1')
ZZ = QubitOperator('Z0 Z1')
II = QubitOperator('')
XY = QubitOperator('X0 Y1')
YX = QubitOperator('Y0 X1')
ZI = QubitOperator('Z0')
IZ = QubitOperator('Z1')

Ucliff = seniority_solving_clifford_operator(Nqubits=2)

print(apply_unitary_product(XX, Ucliff))
print(apply_unitary_product(YY, Ucliff))
print(apply_unitary_product(ZZ, Ucliff))
print(apply_unitary_product(II, Ucliff))
print(apply_unitary_product(XY, Ucliff))
print(apply_unitary_product(YX, Ucliff))
print(apply_unitary_product(ZI, Ucliff))
print(apply_unitary_product(IZ, Ucliff))

0.9999999999999996 [X1]
-0.9999999999999996 [Z0 X1]
0.9999999999999996 [Z0]
0.9999999999999996 []
0.9999999999999996 [Y1]
0.9999999999999996 [Z0 Y1]
0.9999999999999996 [Z0 Z1]
0.9999999999999996 [Z1]


Next, I check if the seniority solving Clifford transformation correctly encodes the seniority configuration on the even-index qubits for a 2N qubit state

In [13]:
# seniority zero state

Nqubits     = 8

psi_source  = uniform(-1, 1, 2 ** (Nqubits // 2))
psi_source  = psi_source / np.linalg.norm(psi_source)
psi_full    = duplicate_statevector_qubit_wise(psi_source)

Ucliff      = get_sparse_operator(compute_product_of_unitaries(seniority_solving_clifford_operator(Nqubits)), Nqubits)

psi_tapered = Ucliff @ psi_full

RED         = '\033[91m'
DEFAULT     = '\033[0m'


for i, val in enumerate(psi_tapered):
    if np.abs(val) > 1e-12:
        bin_list            = decimal_to_binary_list(i, length=Nqubits)
        bin_string          = ''.join([str(z) for z in bin_list])
        coloured_bin_string = ''
        for i, s in enumerate(bin_string):
            if i % 2 == 0:
                coloured_bin_string += f"{RED}{s}{DEFAULT}"
            else:
                coloured_bin_string += s

        print(coloured_bin_string, np.round(val, 4))


00000000 (0.3918+0j)
00000001 (0.0673+0j)
00000100 (-0.2522+0j)
00000101 (-0.1121+0j)
00010000 (-0.1315+0j)
00010001 (-0.2469+0j)
00010100 (0.2172+0j)
00010101 (0.3661+0j)
01000000 (0.1784+0j)
01000001 (-0.2046+0j)
01000100 (0.3406+0j)
01000101 (-0.4121+0j)
01010000 (0.2903+0j)
01010001 (-0.0322+0j)
01010100 (-0.2405+0j)
01010101 (0.0605+0j)


In [14]:
# seniority [0,1,0,0] state

Nqubits     = 8

first       = duplicate_statevector_qubit_wise(uniform(-1, 1, 2))
second      = uniform(-1, 1) * np.array([0,1,0,0]) + uniform(-1, 1) * np.array([0,0,1,0])
third       = duplicate_statevector_qubit_wise(uniform(-1, 1, 4))

psi_full    = np.kron(np.kron(first, second), third)

Ucliff      = get_sparse_operator(compute_product_of_unitaries(seniority_solving_clifford_operator(Nqubits)), Nqubits)

psi_tapered = Ucliff @ psi_full

RED         = '\033[91m'
DEFAULT     = '\033[0m'


for i, val in enumerate(psi_tapered):
    if np.abs(val) > 1e-12:
        bin_list            = decimal_to_binary_list(i, length=Nqubits)
        bin_string          = ''.join([str(z) for z in bin_list])
        coloured_bin_string = ''
        for i, s in enumerate(bin_string):
            if i % 2 == 0:
                coloured_bin_string += f"{RED}{s}{DEFAULT}"
            else:
                coloured_bin_string += s

        print(coloured_bin_string, np.round(val, 4))


00100000 (0.1159+0j)
00100001 (0.1197+0j)
00100100 (-0.2159+0j)
00100101 (-0.2606+0j)
00110000 (-0.1559+0j)
00110001 (-0.161+0j)
00110100 (0.2904+0j)
00110101 (0.3505+0j)
01100000 (0.0121+0j)
01100001 (0.0125+0j)
01100100 (-0.0225+0j)
01100101 (-0.0272+0j)
01110000 (-0.0163+0j)
01110001 (-0.0168+0j)
01110100 (0.0303+0j)
01110101 (0.0366+0j)


Now, I focus on the two main functions of `utils_hamiltonian.py`, which process the Hamiltonian into a modified Hamiltonian for quantum measurements.

The first check is: the matrix element of two seniority zero states is unmodified by the processing of the Hamiltonian

In [15]:
for Nqubits in range(2, 15, 2):
    Nterms      = 1000
    zero_config = [0 for _ in range(Nqubits // 2)]

    # generate random bra and ket

    ket = duplicate_statevector_qubit_wise(uniform(-1, 1, 2 ** (Nqubits // 2)))
    ket = ket / np.linalg.norm(ket)

    bra = duplicate_statevector_qubit_wise(uniform(-1, 1, 2 ** (Nqubits // 2)))
    bra = bra / np.linalg.norm(bra)

    # generate random Hamiltonian and process

    H                 = random_pauli_hamiltonian(Nqubits, Nterms)
    H_sparse          = get_sparse_operator(H, Nqubits)

    Hprocessed        = process_hamiltonian_to_remove_irrelevant_terms(H, Nqubits, zero_config, zero_config)
    Hprocessed_sparse = get_sparse_operator(Hprocessed, Nqubits)

    print(f'''
        Number of qubits              = {Nqubits}
        Number of terms in H          = {len(H.terms)}
        Number of terms in Hprocessed = {len(Hprocessed.terms)}
        <bra|H|ket>                   = {np.round(bra @ H_sparse @ ket, 6)}
        <bra|Hprocessed|ket>          = {np.round(bra @ Hprocessed_sparse @ ket, 6)}
    ''')


        Number of qubits              = 2
        Number of terms in H          = 16
        Number of terms in Hprocessed = 8
        <bra|H|ket>                   = (13.666956-10.491266j)
        <bra|Hprocessed|ket>          = (13.666956-10.491266j)
    

        Number of qubits              = 4
        Number of terms in H          = 252
        Number of terms in Hprocessed = 63
        <bra|H|ket>                   = (-2.257328+7.930307j)
        <bra|Hprocessed|ket>          = (-2.257328+7.930307j)
    

        Number of qubits              = 6
        Number of terms in H          = 876
        Number of terms in Hprocessed = 111
        <bra|H|ket>                   = (-2.008596+3.37571j)
        <bra|Hprocessed|ket>          = (-2.008596+3.37571j)
    

        Number of qubits              = 8
        Number of terms in H          = 992
        Number of terms in Hprocessed = 62
        <bra|H|ket>                   = (2.752056-2.744551j)
        <bra|Hprocessed|ket>     

Now I repeat the same check for seniority non-zero states. Because these are harder to generate as a black-box I will do each example manually

In [16]:
Nqubits    = 8
bra_config = [0,0,0,0]
ket_config = [0,1,0,0]

bra = duplicate_statevector_qubit_wise(uniform(-1, 1, 2 ** (Nqubits // 2)))
bra = bra / np.linalg.norm(bra)

first  = duplicate_statevector_qubit_wise(uniform(-1, 1, 2))
second = uniform(-1, 1) * np.array([0,1,0,0]) + uniform(-1, 1) * np.array([0,0,1,0])
third  = duplicate_statevector_qubit_wise(uniform(-1, 1, 4))

ket = np.kron(np.kron(first, second), third)
ket = ket / np.linalg.norm(ket)

# generate random Hamiltonian and process

H                 = random_pauli_hamiltonian(Nqubits, Nterms)
H_sparse          = get_sparse_operator(H, Nqubits)

Hprocessed        = process_hamiltonian_to_remove_irrelevant_terms(H, Nqubits, bra_config, ket_config)
Hprocessed_sparse = get_sparse_operator(Hprocessed, Nqubits)

print(f'''
    Number of qubits              = {Nqubits}
    Number of terms in H          = {len(H.terms)}
    Number of terms in Hprocessed = {len(Hprocessed.terms)}
    <bra|H|ket>                   = {np.round(bra @ H_sparse @ ket, 6)}
    <bra|Hprocessed|ket>          = {np.round(bra @ Hprocessed_sparse @ ket, 6)}
''')


    Number of qubits              = 8
    Number of terms in H          = 994
    Number of terms in Hprocessed = 59
    <bra|H|ket>                   = (1.314467+0.970523j)
    <bra|Hprocessed|ket>          = (1.314467+0.970523j)



In [ ]:
Nqubits    = 12
bra_config = [0,0,0,0,0,0]
ket_config = [0,1,0,0,0,0]

bra = duplicate_statevector_qubit_wise(uniform(-1, 1, 2 ** (Nqubits // 2)))
bra = bra / np.linalg.norm(bra)

first  = duplicate_statevector_qubit_wise(uniform(-1, 1, 2))
second = uniform(-1, 1) * np.array([0,1,0,0]) + uniform(-1, 1) * np.array([0,0,1,0])
third  = duplicate_statevector_qubit_wise(uniform(-1, 1, 2**4))

ket = np.kron(np.kron(first, second), third)
ket = ket / np.linalg.norm(ket)

# generate random Hamiltonian and process

H                 = random_pauli_hamiltonian(Nqubits, Nterms)
H_sparse          = get_sparse_operator(H, Nqubits)

Hprocessed        = process_hamiltonian_to_remove_irrelevant_terms(H, Nqubits, bra_config, ket_config)
Hprocessed_sparse = get_sparse_operator(Hprocessed, Nqubits)

print(f'''
    Number of qubits              = {Nqubits}
    Number of terms in H          = {len(H.terms)}
    Number of terms in Hprocessed = {len(Hprocessed.terms)}
    <bra|H|ket>                   = {np.round(bra @ H_sparse @ ket, 6)}
    <bra|Hprocessed|ket>          = {np.round(bra @ Hprocessed_sparse @ ket, 6)}
''')


    Number of qubits              = 12
    Number of terms in H          = 1000
    Number of terms in Hprocessed = 14
    <bra|H|ket>                   = (0.233997+0.066835j)
    <bra|Hprocessed|ket>          = (0.233997+0.066835j)



In [18]:
Nqubits    = 12
bra_config = [0,0,0,1,1,0]
ket_config = [0,1,1,0,0,0]

first  = duplicate_statevector_qubit_wise(uniform(-1, 1, 2**3))
second = uniform(-1, 1) * np.array([0,1,0,0]) + uniform(-1, 1) * np.array([0,0,1,0])
third  = uniform(-1, 1) * np.array([0,1,0,0]) + uniform(-1, 1) * np.array([0,0,1,0])
fourth = duplicate_statevector_qubit_wise(uniform(-1, 1, 2**1))

bra = np.kron(np.kron(np.kron(first, second), third), fourth)
bra = bra / np.linalg.norm(bra)

first  = duplicate_statevector_qubit_wise(uniform(-1, 1, 2**1))
second = uniform(-1, 1) * np.array([0,1,0,0]) + uniform(-1, 1) * np.array([0,0,1,0])
third  = uniform(-1, 1) * np.array([0,1,0,0]) + uniform(-1, 1) * np.array([0,0,1,0])
fourth = duplicate_statevector_qubit_wise(uniform(-1, 1, 2**3))

ket = np.kron(np.kron(np.kron(first, second), third), fourth)
ket = ket / np.linalg.norm(ket)

# generate random Hamiltonian and process

H                 = random_pauli_hamiltonian(Nqubits, Nterms)
H_sparse          = get_sparse_operator(H, Nqubits)

Hprocessed        = process_hamiltonian_to_remove_irrelevant_terms(H, Nqubits, bra_config, ket_config)
Hprocessed_sparse = get_sparse_operator(Hprocessed, Nqubits)

print(f'''
    Number of qubits              = {Nqubits}
    Number of terms in H          = {len(H.terms)}
    Number of terms in Hprocessed = {len(Hprocessed.terms)}
    <bra|H|ket>                   = {np.round(bra @ H_sparse @ ket, 6)}
    <bra|Hprocessed|ket>          = {np.round(bra @ Hprocessed_sparse @ ket, 6)}
''')


    Number of qubits              = 12
    Number of terms in H          = 1000
    Number of terms in Hprocessed = 11
    <bra|H|ket>                   = (0.02838-0.419396j)
    <bra|Hprocessed|ket>          = (0.02838-0.419396j)



Now, I also compare with the tapered Hamiltonian. So, below, I check if the original H, the processed H, and the tapered H, all have the same matrix elements for states with fixed seniority configurations. First, I check for zero seniority.

In [10]:
for Nqubits in range(2, 15, 2):
    Nterms      = 1000
    zero_config = [0 for _ in range(Nqubits // 2)]

    bra_N  = uniform(-1, 1, 2 ** (Nqubits // 2))
    bra_N  = bra_N / np.linalg.norm(bra_N)
    bra_2N = duplicate_statevector_qubit_wise(bra_N)
    bra_2N = bra_2N / np.linalg.norm(bra_2N)

    ket_N  = uniform(-1, 1, 2 ** (Nqubits // 2))
    ket_N  = ket_N / np.linalg.norm(ket_N)
    ket_2N = duplicate_statevector_qubit_wise(ket_N)
    ket_2N = ket_2N / np.linalg.norm(ket_2N)

    H          = random_pauli_hamiltonian(Nqubits, Nterms)
    Hprocessed = process_hamiltonian_to_remove_irrelevant_terms(H, Nqubits, zero_config, zero_config)
    Htapered   = process_hamiltonian_to_project_onto_symmetric_subspace(H, Nqubits, zero_config, zero_config)

    H_sparse          = get_sparse_operator(H, Nqubits)
    Hprocessed_sparse = get_sparse_operator(Hprocessed, Nqubits)
    Htapered_sparse   = get_sparse_operator(Htapered, Nqubits // 2)


    print(f'''
        Number of qubits              = {Nqubits}
        Number of terms in H          = {len(H.terms)}
        Number of terms in Hprocessed = {len(Hprocessed.terms)}
        Number of terms in Htapered   = {len(Htapered.terms)}
        <bra|H|ket>                   = {np.round(bra_2N @ H_sparse @ ket_2N, 6)}
        <bra|Hprocessed|ket>          = {np.round(bra_2N @ Hprocessed_sparse @ ket_2N, 6)}
        <bra|Htapered|ket>            = {np.round(bra_N @ Htapered_sparse @ ket_N, 6)}
    ''')


        Number of qubits              = 2
        Number of terms in H          = 16
        Number of terms in Hprocessed = 8
        Number of terms in Htapered   = 4
        <bra|H|ket>                   = (5.325116-3.342907j)
        <bra|Hprocessed|ket>          = (5.325116-3.342907j)
        <bra|Htapered|ket>            = (5.325116-3.342907j)
    

        Number of qubits              = 4
        Number of terms in H          = 253
        Number of terms in Hprocessed = 63
        Number of terms in Htapered   = 16
        <bra|H|ket>                   = (-6.624199-6.619782j)
        <bra|Hprocessed|ket>          = (-6.624199-6.619782j)
        <bra|Htapered|ket>            = (-6.624199-6.619782j)
    

        Number of qubits              = 6
        Number of terms in H          = 877
        Number of terms in Hprocessed = 119
        Number of terms in Htapered   = 57
        <bra|H|ket>                   = (-1.880751+1.662575j)
        <bra|Hprocessed|ket>          = (-

In [14]:
for _ in range(10):
    Nqubits     = 12
    Nterms      = 1000
    bra_config  = [0,1,0,0,0,0]
    ket_config  = [0,1,0,0,0,0]

    # make bra -> for now assume the ket is the same

    one_N    = uniform(-1, 1, 2**1)
    one_2N   = duplicate_statevector_qubit_wise(one_N)

    two_N    = uniform(-1, 1) * np.array([0,1]) + uniform(-1, 1) * np.array([1,0])
    two_2N   = two_N[0] * np.array([0,0,1,0]) + two_N[1] * np.array([0,1,0,0])

    three_N  = uniform(-1, 1, 2**4)
    three_2N = duplicate_statevector_qubit_wise(three_N)

    bra_N    = np.kron(np.kron(one_N, two_N), three_N)
    bra_2N   = np.kron(np.kron(one_2N, two_2N), three_2N)

    ket_N    = bra_N.copy()
    ket_2N   = bra_2N.copy()

    H          = random_pauli_hamiltonian(Nqubits, Nterms)
    Hprocessed = process_hamiltonian_to_remove_irrelevant_terms(H, Nqubits, bra_config, ket_config)
    Htapered   = process_hamiltonian_to_project_onto_symmetric_subspace(H, Nqubits, bra_config, ket_config)

    H_sparse          = get_sparse_operator(H, Nqubits)
    Hprocessed_sparse = get_sparse_operator(Hprocessed, Nqubits)
    Htapered_sparse   = get_sparse_operator(Htapered, Nqubits // 2)


    print(f'''
        Number of qubits              = {Nqubits}
        Number of terms in H          = {len(H.terms)}
        Number of terms in Hprocessed = {len(Hprocessed.terms)}
        Number of terms in Htapered   = {len(Htapered.terms)}
        <bra|H|ket>                   = {np.round(bra_2N @ H_sparse @ ket_2N, 6)}
        <bra|Hprocessed|ket>          = {np.round(bra_2N @ Hprocessed_sparse @ ket_2N, 6)}
        <bra|Htapered|ket>            = {np.round(bra_N @ Htapered_sparse @ ket_N, 6)}
    ''')


        Number of qubits              = 12
        Number of terms in H          = 1000
        Number of terms in Hprocessed = 15
        Number of terms in Htapered   = 15
        <bra|H|ket>                   = (0.001909-0j)
        <bra|Hprocessed|ket>          = (0.001909+0j)
        <bra|Htapered|ket>            = (0.001909-0j)
    

        Number of qubits              = 12
        Number of terms in H          = 1000
        Number of terms in Hprocessed = 15
        Number of terms in Htapered   = 15
        <bra|H|ket>                   = (0.009475+0j)
        <bra|Hprocessed|ket>          = (0.009475+0j)
        <bra|Htapered|ket>            = (0.009475+0j)
    

        Number of qubits              = 12
        Number of terms in H          = 1000
        Number of terms in Hprocessed = 10
        Number of terms in Htapered   = 10
        <bra|H|ket>                   = (-0.006515-0j)
        <bra|Hprocessed|ket>          = (-0.006515-0j)
        <bra|Htapered|ket>     

In [43]:
for _ in range(10):
    Nqubits    = 12
    Nterms     = 5000
    bra_config = [0,0,1,1,0,0]
    ket_config = [0,1,0,1,0,0]

    one_N    = uniform(-1, 1, 2**2)
    one_2N   = duplicate_statevector_qubit_wise(one_N)

    two_N    = uniform(-1, 1) * np.array([0,1]) + uniform(-1, 1) * np.array([1,0])
    two_2N   = two_N[0] * np.array([0,0,1,0]) + two_N[1] * np.array([0,1,0,0])

    three_N  = uniform(-1, 1) * np.array([0,1]) + uniform(-1, 1) * np.array([1,0])
    three_2N = three_N[0] * np.array([0,0,1,0]) + three_N[1] * np.array([0,1,0,0])

    four_N   = uniform(-1, 1, 2**2)
    four_2N  = duplicate_statevector_qubit_wise(four_N)

    bra_N  = np.kron(np.kron(np.kron(one_N, two_N), three_N), four_N)
    bra_N  = bra_N / np.linalg.norm(bra_N)
    bra_2N = np.kron(np.kron(np.kron(one_2N, two_2N), three_2N), four_2N)
    bra_2N = bra_2N / np.linalg.norm(bra_2N)

    one_N      = uniform(-1, 1, 2**1)
    one_2N     = duplicate_statevector_qubit_wise(one_N)

    two_N      = uniform(-1, 1) * np.array([0,1]) + uniform(-1, 1) * np.array([1,0])
    two_2N     = two_N[0] * np.array([0,0,1,0]) + two_N[1] * np.array([0,1,0,0])

    three_N    = uniform(-1, 1, 2**1)
    three_2N   = duplicate_statevector_qubit_wise(three_N)

    four_N    = uniform(-1, 1) * np.array([0,1]) + uniform(-1, 1) * np.array([1,0])
    four_2N   = four_N[0] * np.array([0,0,1,0]) + four_N[1] * np.array([0,1,0,0])

    five_N    = uniform(-1, 1, 2**2)
    five_2N   = duplicate_statevector_qubit_wise(five_N)

    ket_N  = np.kron(np.kron(np.kron(np.kron(one_N, two_N), three_N), four_N), five_N)
    ket_N  = ket_N / np.linalg.norm(ket_N)
    ket_2N = np.kron(np.kron(np.kron(np.kron(one_2N, two_2N), three_2N), four_2N), five_2N)
    ket_2N = ket_2N / np.linalg.norm(ket_2N)

    H          = random_pauli_hamiltonian(Nqubits, Nterms)
    Hprocessed = process_hamiltonian_to_remove_irrelevant_terms(H, Nqubits, bra_config, ket_config)
    Htapered   = process_hamiltonian_to_project_onto_symmetric_subspace(H, Nqubits, bra_config, ket_config)

    H_sparse          = get_sparse_operator(H, Nqubits)
    Hprocessed_sparse = get_sparse_operator(Hprocessed, Nqubits)
    Htapered_sparse   = get_sparse_operator(Htapered, Nqubits // 2)

    print(f'''
        Number of qubits              = {Nqubits}
        Number of terms in H          = {len(H.terms)}
        Number of terms in Hprocessed = {len(Hprocessed.terms)}
        Number of terms in Htapered   = {len(Htapered.terms)}
        <bra|H|ket>                   = {np.round(bra_2N @ H_sparse @ ket_2N, 6)}
        <bra|Hprocessed|ket>          = {np.round(bra_2N @ Hprocessed_sparse @ ket_2N, 6)}
        <bra|Htapered|ket>            = {np.round(bra_N @ Htapered_sparse @ ket_N, 6)}
    ''')


        Number of qubits              = 12
        Number of terms in H          = 5000
        Number of terms in Hprocessed = 63
        Number of terms in Htapered   = 62
        <bra|H|ket>                   = (-0.383164-0.56669j)
        <bra|Hprocessed|ket>          = (-0.383164-0.56669j)
        <bra|Htapered|ket>            = (-0.383164-0.56669j)
    

        Number of qubits              = 12
        Number of terms in H          = 5000
        Number of terms in Hprocessed = 87
        Number of terms in Htapered   = 86
        <bra|H|ket>                   = (1.012129+0.185437j)
        <bra|Hprocessed|ket>          = (1.012129+0.185437j)
        <bra|Htapered|ket>            = (1.012129+0.185437j)
    

        Number of qubits              = 12
        Number of terms in H          = 4999
        Number of terms in Hprocessed = 92
        Number of terms in Htapered   = 91
        <bra|H|ket>                   = (-0.124062-1.111977j)
        <bra|Hprocessed|ket>         